# Preprocessing Pipeline

**Objective**: Prepare the engineered dataset for modelling. Runs once — all model notebooks load from the outputs saved here.

**Steps:**
1. Load `factory_engineered.csv`
2. Define feature and target columns
3. Train/test split (80/20 random)
4. Target encode `Planning Area` (fit on train only)
5. One-hot encode `Region`, `Floor Level`, `Type of Sale`
6. Save `X_train`, `X_test`, `y_train`, `y_test` to `data/processed/`
7. Save encoders to `models/`

---

## 1. Imports & Data Loading

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("../data/processed/factory_engineered.csv")

os.makedirs("../models", exist_ok=True)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Dataset shape: (3782, 22)
Columns: ['Contract Date', 'Project Name', 'Street Name', 'Type Of Area', 'Floor Level', 'Unit Price ($ psf)', 'Type of Sale', 'Region', 'Planning Area', 'Remaining_Lease_Years', 'GDP_YoY_Growth_Rate', 'CPI_All_Items', 'Unemployment_Rate', 'Price_Index', '10Y_Bond_Yield', 'SORA_3M_Compounded', 'Cement_Bulk_Per_Tonne', 'Steel_Rebar_Per_Tonne', 'Log_Unit_Price', 'Log_Area', 'Lease_Remaining_Ratio', 'dist_to_mrt_m']


## 2. Define Feature & Target Columns

**Excluded from features:**
- `Price_Index`, `Steel_Rebar_Per_Tonne` — removed by VIF (notebook 03)
- `Contract Date`, `Project Name`, `Street Name` — identifiers, not predictive
- `Type Of Area` — near-constant (99.8% Strata)
- `Unit Price ($ psf)`, `Log_Unit_Price` — targets, not features

**Targets:**
- `Unit Price ($ psf)` — for tree-based models
- `Log_Unit_Price` — for linear models (back-transform via exp() after prediction)

In [2]:
# Numerical features
NUMERICAL_COLS = [
    "Log_Area",
    "Remaining_Lease_Years",
    "Lease_Remaining_Ratio",
    "dist_to_mrt_m",
    "GDP_YoY_Growth_Rate",
    "CPI_All_Items",
    "Unemployment_Rate",
    "10Y_Bond_Yield",
    "SORA_3M_Compounded",
    "Cement_Bulk_Per_Tonne"
]

# Categorical features
TARGET_ENCODE_COL = "Planning Area"       # target encoding — high cardinality (24 levels)
ONEHOT_COLS = ["Region", "Floor Level", "Type of Sale"]  # one-hot — low cardinality

# Targets
TARGET_RAW = "Unit Price ($ psf)"         # tree-based models
TARGET_LOG = "Log_Unit_Price"             # linear models

print(f"Numerical features:  {len(NUMERICAL_COLS)}")
print(f"Target encode:       {TARGET_ENCODE_COL} ({df[TARGET_ENCODE_COL].nunique()} levels)")
print(f"One-hot encode:      {ONEHOT_COLS}")
print(f"Targets:             {TARGET_RAW} | {TARGET_LOG}")

Numerical features:  10
Target encode:       Planning Area (24 levels)
One-hot encode:      ['Region', 'Floor Level', 'Type of Sale']
Targets:             Unit Price ($ psf) | Log_Unit_Price


## 3. Train/Test Split

**80/20 random split** — appropriate for cross-sectional regression (not time series forecasting).

Split is done **before encoding** to prevent data leakage — target encoding must be fit on training data only.

In [3]:
# All feature columns (pre-encoding)
FEATURE_COLS_RAW = NUMERICAL_COLS + [TARGET_ENCODE_COL] + ONEHOT_COLS

X = df[FEATURE_COLS_RAW].copy()
y_raw = df[TARGET_RAW].copy()
y_log = df[TARGET_LOG].copy()

X_train, X_test, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X, y_raw, y_log, test_size=0.2, random_state=42
)

print(f"Train size: {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test size:  {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTrain target — Mean: ${y_train_raw.mean():.2f} | Median: ${y_train_raw.median():.2f}")
print(f"Test target  — Mean: ${y_test_raw.mean():.2f} | Median: ${y_test_raw.median():.2f}")

Train size: 3025 rows (80.0%)
Test size:  757 rows (20.0%)

Train target — Mean: $442.15 | Median: $437.00
Test target  — Mean: $439.87 | Median: $430.00


## 4. Target Encode `Planning Area`

Replace each Planning Area with its mean unit price, computed **on training data only**.

**Why fit on train only**: Computing the mean on the full dataset leaks test price information into the encoding — the model indirectly sees test target values during training.

**Rare area handling**: Planning areas with very few transactions (e.g. Paya Lebar = 1) have unreliable mean estimates. Use global training mean as fallback for unseen areas.

In [4]:
# Compute mean price per planning area on TRAIN only
target_encode_map = X_train.join(y_train_raw).groupby(TARGET_ENCODE_COL)[TARGET_RAW].mean()
global_mean = y_train_raw.mean()

print("Planning Area target encoding (mean Unit Price $ psf from train):")
print(target_encode_map.sort_values(ascending=False).round(2).to_string())
print(f"\nGlobal mean (fallback for unseen areas): ${global_mean:.2f}")

# Apply encoding to train and test
X_train["Planning_Area_Encoded"] = X_train[TARGET_ENCODE_COL].map(target_encode_map).fillna(global_mean)
X_test["Planning_Area_Encoded"] = X_test[TARGET_ENCODE_COL].map(target_encode_map).fillna(global_mean)

# Save encoder map
joblib.dump({"map": target_encode_map, "global_mean": global_mean}, "../models/target_encoder.pkl")
print("\nTarget encoder saved to models/target_encoder.pkl")

Planning Area target encoding (mean Unit Price $ psf from train):
Planning Area
Kallang         984.43
Toa Payoh       739.78
Bukit Merah     612.79
Bishan          600.14
Serangoon       552.40
Geylang         530.00
Ang Mo Kio      504.84
Clementi        463.16
Bedok           436.64
Yishun          428.48
Woodlands       427.77
Bukit Batok     413.76
Queenstown      365.70
Tampines        349.28
Tuas            327.23
Sungei Kadut    325.71
Sembawang       321.79
Jurong West     321.16
Boon Lay        229.18
Changi          208.60
Pasir Ris       205.81
Jurong East     186.67
Pioneer         148.40

Global mean (fallback for unseen areas): $442.15

Target encoder saved to models/target_encoder.pkl


## 5. One-Hot Encode `Region`, `Floor Level`, `Type of Sale`

Fit on training data only, apply to both train and test.

**`handle_unknown='ignore'`**: If test contains an unseen category, encode as all zeros rather than raising an error.

In [5]:
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
ohe.fit(X_train[ONEHOT_COLS])

# Transform train
ohe_train = pd.DataFrame(
    ohe.transform(X_train[ONEHOT_COLS]),
    columns=ohe.get_feature_names_out(ONEHOT_COLS),
    index=X_train.index
)

# Transform test
ohe_test = pd.DataFrame(
    ohe.transform(X_test[ONEHOT_COLS]),
    columns=ohe.get_feature_names_out(ONEHOT_COLS),
    index=X_test.index
)

# Save encoder
joblib.dump(ohe, "../models/onehot_encoder.pkl")

print("One-hot encoded columns:")
for col in ohe.get_feature_names_out(ONEHOT_COLS):
    print(f"  {col}")
print(f"\nOne-hot encoder saved to models/onehot_encoder.pkl")

One-hot encoded columns:
  Region_East Region
  Region_North Region
  Region_North-East Region
  Region_West Region
  Floor Level_Non-First Floor
  Floor Level_Unknown
  Type of Sale_Resale

One-hot encoder saved to models/onehot_encoder.pkl


## 6. Assemble Final Feature Matrix

In [6]:
# Drop original categorical columns — replaced by encoded versions
DROP_CAT = [TARGET_ENCODE_COL] + ONEHOT_COLS

X_train_final = pd.concat([X_train.drop(columns=DROP_CAT), ohe_train], axis=1)
X_test_final = pd.concat([X_test.drop(columns=DROP_CAT), ohe_test], axis=1)

print(f"Final X_train shape: {X_train_final.shape}")
print(f"Final X_test shape:  {X_test_final.shape}")
print(f"\nFinal feature columns ({len(X_train_final.columns)}):")
for col in X_train_final.columns:
    print(f"  {col}")

Final X_train shape: (3025, 18)
Final X_test shape:  (757, 18)

Final feature columns (18):
  Log_Area
  Remaining_Lease_Years
  Lease_Remaining_Ratio
  dist_to_mrt_m
  GDP_YoY_Growth_Rate
  CPI_All_Items
  Unemployment_Rate
  10Y_Bond_Yield
  SORA_3M_Compounded
  Cement_Bulk_Per_Tonne
  Planning_Area_Encoded
  Region_East Region
  Region_North Region
  Region_North-East Region
  Region_West Region
  Floor Level_Non-First Floor
  Floor Level_Unknown
  Type of Sale_Resale


## 7. Save Outputs

In [7]:
# Save feature matrices
X_train_final.to_csv("../data/processed/X_train.csv", index=False)
X_test_final.to_csv("../data/processed/X_test.csv", index=False)

# Save targets — both raw and log in one file
y_train = pd.DataFrame({"Unit Price ($ psf)": y_train_raw, "Log_Unit_Price": y_train_log})
y_test = pd.DataFrame({"Unit Price ($ psf)": y_test_raw, "Log_Unit_Price": y_test_log})

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Saved:")
print("  data/processed/X_train.csv")
print("  data/processed/X_test.csv")
print("  data/processed/y_train.csv  (columns: Unit Price $ psf, Log_Unit_Price)")
print("  data/processed/y_test.csv   (columns: Unit Price $ psf, Log_Unit_Price)")
print("  models/target_encoder.pkl")
print("  models/onehot_encoder.pkl")

print(f"""
Summary:
  Train rows:     {X_train_final.shape[0]}
  Test rows:      {X_test_final.shape[0]}
  Features:       {X_train_final.shape[1]}
  Target (raw):   Unit Price ($ psf) — use for tree-based models
  Target (log):   Log_Unit_Price     — use for linear models, back-transform via exp()
""")

Saved:
  data/processed/X_train.csv
  data/processed/X_test.csv
  data/processed/y_train.csv  (columns: Unit Price $ psf, Log_Unit_Price)
  data/processed/y_test.csv   (columns: Unit Price $ psf, Log_Unit_Price)
  models/target_encoder.pkl
  models/onehot_encoder.pkl

Summary:
  Train rows:     3025
  Test rows:      757
  Features:       18
  Target (raw):   Unit Price ($ psf) — use for tree-based models
  Target (log):   Log_Unit_Price     — use for linear models, back-transform via exp()

